# Hyperparameter Tuning using Optuna

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.model_selection import cross_val_score
import optuna
from optuna.trial import Trial

In [ ]:
# Check the ability of GPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataloaders

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="../../datasets/",
    train = True,
    download = True,
    transform=transform,
)

test_data = datasets.FashionMNIST(
    root="../../datasets/",
    train=False,
    download=True,
    transform=transform
)


In [ ]:
# Model Architecture: For optuna objective function
class OptunaANN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, droptout_rate):
        super().__init__() # Must always do
        layers = []
        layers.append(nn.Flatten())
        for i in range(num_hidden_layers):
            # Limitations: All hidden layers will have same number of neurons
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(droptout_rate))
            input_dim = neurons_per_layer # Clever way to make previous layer's output, current layer's input
        layers.append(nn.Linear(neurons_per_layer, output_dim)) 
        self.model = nn.Sequential(
            *layers # Unpacked layer
        )
        
        
    def forward(self, x):
        
        return self.model(x)
        

In [ ]:
# Create Objective Function
def objective(trial:Trial):
    
    # Next Hyperparameter values from search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 10, 30, step=5)
    learning_rate = trial.suggest_float("learning_rate", 10e-5, 10e-1, log=True)
        # Learning rate will be in logarithmic scale   
    dropout_rate = trial.suggest_float("droptout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True) # Logarithmic scale here as well
    
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_data, batch_size=64, shuffle=False, pin_memory=True)
    
    # Model initialization
    input_dim = 28 * 28
    output_dim = 10
    
    model = OptunaANN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device=device)
    

    # Optimizer selection
    criterion = nn.CrossEntropyLoss()
    optimizers = {
        'Adam': torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        'SGD': torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        'RMSprop': torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    }
    optimizer = optimizers.get(optimizer_name, optimizers['SGD'])
    

    # Training loop
    for epoch in range(epochs):
        batch_features: torch.Tensor
        batch_labels: torch.Tensor
        for batch_features, batch_labels in train_loader:
            
            # Move batch_features and batch_labels to GPU
            batch_features, batch_labels = batch_features.to(device=device), batch_labels.to(device=device)
            
            # Forward pass
            outputs = model(batch_features)
            
            # Calculate Loss
            loss = criterion(outputs, batch_labels)

            # Clear gradients
            optimizer.zero_grad()
            
            # Backward Pass
            loss.backward()
            
            # Update Gradients
            optimizer.step()
            
    # Evaluate
    with torch.no_grad():
        total = 0
        correct = 0
        for batch_features, batch_labels in test_loader:
            
            # Move data to GPU
            batch_features, batch_labels = batch_features.to(device=device), batch_labels.to(device=device)
            
            outputs = model(batch_features)
            
            _, predicted = torch.max(outputs, 1)
            
            total = total + batch_labels.shape[0]
            correct = correct + (predicted == batch_labels).sum().item()
         
    accuracy = correct/total
    
    return accuracy

In [ ]:
study = optuna.create_study(direction='maximize')

In [ ]:
study.optimize(objective, n_trials=20)

In [ ]:
study.best_params, study.best_value